# Demo 08 : Debate and selection

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


We use the made-up Veenhaven Bus records. We count every attempted model call, input and output tokens, and elapsed seconds, which include the wait in the shared rate limiter.

In [2]:
import json
import operator
import re
import sqlite3
import threading
import time
from pathlib import Path
from typing import Annotated, Literal
from typing_extensions import NotRequired, TypedDict

from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import wrap_model_call
from langchain.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain.tools import ToolRuntime, tool
from langchain_core.callbacks import BaseCallbackHandler
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import Command, Send
from pydantic import BaseModel, Field


class Counter(BaseCallbackHandler):
    def __init__(self):
        self.lock = threading.Lock()
        self.reset()

    def reset(self):
        self.calls = self.input = self.output = self.invalid = self.errors = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        with self.lock:
            self.calls += 1

    def on_llm_end(self, response, **kwargs):
        with self.lock:
            for generations in response.generations:
                for generation in generations:
                    message = getattr(generation, "message", None)
                    usage = getattr(message, "usage_metadata", None) or {}
                    self.input += usage.get("input_tokens", 0)
                    self.output += usage.get("output_tokens", 0)
                    self.invalid += len(getattr(message, "invalid_tool_calls", []) or [])

    def on_llm_error(self, error, **kwargs):
        with self.lock:
            self.errors += 1


counter = Counter()
llm = make_llm(callbacks=[counter], max_tokens=1200, max_retries=0)
metrics = []


def measured(label, operation):
    counter.reset()
    start = time.perf_counter()
    try:
        return operation()
    finally:
        row = dict(pattern=label, calls=counter.calls, input_tokens=counter.input,
                   output_tokens=counter.output, seconds=round(time.perf_counter() - start, 1),
                   malformed=counter.invalid, errors=counter.errors)
        metrics.append(row)
        print("metrics :", json.dumps(row))


DB = Path(data("demo-world/service.db")).resolve()
MANUALS = Path(data_folder("demo-world/manuals"))
PAGES = {p.name: p.read_text(encoding="utf-8") for p in sorted(MANUALS.glob("*.md"))}


@tool
def part_stock(part: str) -> str:
    """Look up the stock, location and name of a Veenhaven Bus part number."""
    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as db:
        rows = db.execute("SELECT part_no, name, stock, depot FROM parts WHERE part_no = ?",
                          (part.upper(),)).fetchall()
    return json.dumps(rows)


@tool
def fault_guidance(error_code: str) -> str:
    """Read the manual's meaning and first action for a Veenhaven Bus error code."""
    matches = [f"[{name}]\n{line}" for name, text in PAGES.items()
               for line in text.splitlines() if line.startswith("| " + error_code.upper() + " |")]
    return "\n".join(matches) or "No matching error code."


@tool
def run_sql(query: str) -> str:
    """Run a read-only SELECT. Tables: buses(fleet_no, model, depot_id, commissioned, firmware),
    depots(depot_id, name, city, contract), parts(part_no, model, name, stock, price_eur, depot),
    work_orders(wo_id, fleet_no, opened, status, error_code, summary, mechanic_id)."""
    if not query.lstrip().lower().startswith("select"):
        return "Only SELECT statements are allowed."
    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as db:
        return json.dumps(db.execute(query).fetchmany(30))


@tool
def search_manuals(query: str) -> str:
    """Find relevant Veenhaven Bus manuals and bulletins with their file names."""
    words = set(query.lower().split())
    ranked = sorted(PAGES, key=lambda name: -sum(PAGES[name].lower().count(w) for w in words))[:3]
    return "\n\n".join(f"[{name}]\n{PAGES[name]}" for name in ranked)


QUESTION = ("Bus CE12-2021-0117 shows F-CE-110. How many CE-CF-02 cooling fans are in stock, "
            "at which location, and what are the manual's first action and replacement condition for this error? Be brief.")
STOCK_TASK = "How many CE-CF-02 cooling fans are in stock and at which location?"
MANUAL_TASK = "What are the meaning, first action and replacement condition for F-CE-110? Cite the manual."
LIMIT = {"recursion_limit": 24}


def ask(question):
    return {"messages": [HumanMessage(question)]}


def last_text(result):
    return str(next(m.content for m in reversed(result["messages"])
                    if isinstance(m, AIMessage) and m.content and not m.tool_calls))


def agent_by_hand(model, tools, prompt, state_schema=MessagesState):
    bound = model.bind_tools(tools)

    def model_node(state):
        return {"messages": [bound.invoke([SystemMessage(prompt), *state["messages"]])]}

    graph = StateGraph(state_schema)
    graph.add_node("model", model_node)
    graph.add_node("tools", ToolNode(tools))
    graph.add_edge(START, "model")
    graph.add_conditional_edges("model", tools_condition)
    graph.add_edge("tools", "model")
    return graph.compile()


def specialists(model):
    return {
        "parts": agent_by_hand(model, [part_stock], "Answer stock questions from the tool. Return every fact briefly."),
        "manuals": agent_by_hand(model, [fault_guidance], "Answer error-code questions from the manual tool. Cite its file."),
    }


def compose(model, question, findings):
    return model.invoke("Answer from these findings only. Preserve counts, locations and conditions.\n"
                        + question + "\n" + "\n".join(findings)).content


def completeness(answer):
    text = answer.lower()
    return {"stock": bool(re.search(r"\b(6|six)\b", text)) and "rietpoort" in text,
            "action": "fan" in text and "airflow" in text and bool(re.search(r"\b(80|eighty)\b", text))}

## a · debate-and-selection

We ask one agent to argue for inspection first and another for replacement first, on the same evidence. A judge sees both proposals and the source. We inspect whether it retains the manual's replacement condition.

In [3]:
class Review(BaseModel):
    approved: bool
    feedback: str


class DraftState(TypedDict):
    task: str
    draft: str
    feedback: str
    approved: bool
    rounds: int


EVIDENCE = part_stock.invoke({"part": "CE-CF-02"}) + "\n" + part_stock.invoke({"part": "CE-DA-03"})
TASK = "Write a parts note naming CE-CF-02 and CE-DA-03, their stock counts and locations. Mark zero stock as 'out of stock'."


def check_note(draft):
    problems = []
    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as db:
        expected = db.execute("SELECT part_no, stock, depot FROM parts WHERE part_no IN (?, ?)",
                              ("CE-CF-02", "CE-DA-03")).fetchall()
    for part, count, location in expected:
        lines = [line.lower() for line in draft.splitlines() if part.lower() in line.lower()]
        if not lines:
            problems.append("Include " + part)
        elif not any(re.search(r"\b" + str(count) + r"\b", line) and location.lower() in line
                     and (count != 0 or "out of stock" in line) for line in lines):
            problems.append(f"Put {part}, stock {count}, {location}" + (", and 'out of stock'" if count == 0 else "") + " on one line.")
    return problems


def maker_checker(model, code_check=False):
    def make(state):
        draft = model.invoke(state["task"] + "\nEvidence:\n" + EVIDENCE
                             + "\nFeedback:\n" + state["feedback"] + "\nPrevious draft:\n" + state["draft"]).content
        return {"draft": draft, "rounds": state["rounds"] + 1}

    def check(state):
        if code_check:
            problems = check_note(state["draft"])
            approved, feedback = not problems, "\n".join(problems)
        else:
            review = model.with_structured_output(Review).invoke(
                "Check that both named parts, their counts and locations match the evidence. "
                "Zero stock needs 'out of stock'.\nEvidence:\n" + EVIDENCE + "\nDraft:\n" + state["draft"])
            approved, feedback = review.approved, review.feedback
        print("review :", state["rounds"], approved, feedback)
        return {"approved": approved, "feedback": feedback}

    graph = StateGraph(DraftState)
    graph.add_node("make", make)
    graph.add_node("check", check)
    graph.add_edge(START, "make")
    graph.add_edge("make", "check")
    graph.add_conditional_edges("check", lambda s: END if s["approved"] or s["rounds"] >= 2 else "make", ["make", END])
    return graph.compile()


def draft_input():
    return {"task": TASK, "draft": "", "feedback": "", "approved": False, "rounds": 0}

In [4]:
class SelectionState(TypedDict):
    task: str
    candidates: Annotated[list[dict], operator.add]
    answer: str


class Judgement(BaseModel):
    selected: Literal["inspect", "replace"]
    reason: str


def debate_graph(model):
    evidence = fault_guidance.invoke({"error_code": "F-CE-110"}) + "\n" + part_stock.invoke({"part": "CE-CF-02"})

    def propose(role):
        return lambda s: {"candidates": [{"role": role, "text": model.invoke(
            f"Assess the case for {role} first. Use the evidence, retain its conditions, do not invent tests.\n"
            + s["task"] + "\n" + evidence).content}]}

    def judge(state):
        candidates = sorted(state["candidates"], key=lambda c: c["role"])
        verdict = model.with_structured_output(Judgement).invoke(
            "Choose the supported first action. Availability alone does not justify replacement.\nEvidence:\n"
            + evidence + "\nCandidates:\n" + json.dumps(candidates))
        return {"answer": verdict.selected + " : " + verdict.reason}

    graph = StateGraph(SelectionState)
    for role in ["inspect", "replace"]:
        graph.add_node(role, propose(role))
        graph.add_edge(START, role)
    graph.add_node("judge", judge)
    graph.add_edge(["inspect", "replace"], "judge")
    graph.add_edge("judge", END)
    return graph.compile()


def generate_filter_graph(model):
    def generate(state):
        text = model.invoke(TASK + " Use one line per part and digits for counts.\n"
                            + state["task"] + "\nEvidence:\n" + EVIDENCE).content
        return {"candidates": [{"role": state["task"], "text": text}]}

    def select(state):
        checked = [(c, check_note(c["text"])) for c in state["candidates"]]
        for candidate, problems in checked:
            print("filter :", candidate["role"], problems)
        valid = [c for c, problems in checked if not problems]
        chosen = min(valid, key=lambda c: (len(c["text"]), c["role"])) if valid else None
        return {"answer": chosen["text"] if chosen else "No candidate passed the factual checks."}

    graph = StateGraph(SelectionState)
    graph.add_node("generate", generate)
    graph.add_node("select", select)
    graph.add_conditional_edges(START, lambda s: [Send("generate", {"task": style}) for style in ["Compact bullets.", "A compact table."]], ["generate"])
    graph.add_edge("generate", "select")
    graph.add_edge("select", END)
    return graph.compile()

In [5]:
result = measured("debate with judge", lambda: debate_graph(llm).invoke({"task": QUESTION, "candidates": []}, LIMIT))
for candidate in sorted(result["candidates"], key=lambda c: c["role"]):
    print("proposal :", candidate["role"], candidate["text"])
print("judge :", result["answer"])

metrics : {"pattern": "debate with judge", "calls": 3, "input_tokens": 529, "output_tokens": 156, "seconds": 8.6, "malformed": 0, "errors": 0}
proposal : inspect There are **6 CE-CF-02 traction cooling fans** in stock at **Rietpoort**. For F-CE-110, the manual says to **check the traction cooling fan** and **replace it if airflow is below 80 percent**.
proposal : replace There are **6 CE-CF-02 fans** in stock at **Rietpoort**. For F-CE-110, the manual says to **check the traction cooling fan first** and **replace it if airflow is below 80%**.
judge : inspect : The manual requires checking airflow first and replacing the fan only if it is below 80 percent. Stock availability does not establish that condition.


## b · generate-and-filter

We generate alternative notes independently, reject those that fail factual checks, and choose the shortest surviving note in code. We report an empty selection when no candidate passes.

In [6]:
result = measured("generate and filter", lambda: generate_filter_graph(llm).invoke({"task": TASK, "candidates": []}, LIMIT))
for candidate in result["candidates"]:
    print("candidate :", candidate["role"], candidate["text"])
print("selected :", result["answer"])

filter : Compact bullets. []
filter : A compact table. []
metrics : {"pattern": "generate and filter", "calls": 2, "input_tokens": 191, "output_tokens": 96, "seconds": 5.0, "malformed": 0, "errors": 0}
candidate : Compact bullets. - CE-CF-02 — traction cooling fan: 6 in stock; Rietpoort
- CE-DA-03 — door actuator: 0 (out of stock); Rietpoort
candidate : A compact table. | Part | Stock | Location |
|---|---:|---|
| CE-CF-02 | 6 | Rietpoort |
| CE-DA-03 | 0 (out of stock) | Rietpoort |
selected : - CE-CF-02 — traction cooling fan: 6 in stock; Rietpoort
- CE-DA-03 — door actuator: 0 (out of stock); Rietpoort


We can combine these shapes with a bounded tournament or a repair loop. More proposals spend more tokens, and agreement between models does not replace evidence. The examples and code are ours ; the orchestration names are discussed in the [Academy delegation material](https://github.com/langchain-ai/lca-deepagents).

## Things to try

Optional : we can change the question, inspect each context, and compare the answer checks with the call and token counts.